# CNN Fine Tuning - ShuffleNetv2

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torchvision import models, transforms, datasets
from torch.utils.data import DataLoader, random_split, Dataset
from torchvision.models.shufflenetv2 import ShuffleNet_V2_X1_0_Weights
import matplotlib.pyplot as plt
import numpy as np
import os

# Decompress the dataset
This block will decompress the dataset from the zip file. Used for uploading the dataset to Google Colab.

In [ ]:
# import zipfile
# import os

# zip_path = "data.zip"
# extract_path = ""
# with zipfile.ZipFile(zip_path, "r") as zip_ref:
#     zip_ref.extractall(extract_path)

In [ ]:
model_params = {
    "v8-dropout-all": {
        "learning_rate": 0.0001,
        "optimizer_alg": "Adam",
        "weight_decay": 0.00,
        "criterion": "CrossEntropyLoss",
        "batch_size": 64,
        "num_workers": 8,
        "num_epochs": 65,
        "freeze_layers": True,
        "frozen_layers": ["conv1",
                          "stage2"],
        "dropout": True,
        "dropout_prob": {
            "fc": 0.1,
            "conv5": 0.1,
            "stages": {
                "stage4": 0.01,
                "stage3": 0.01,
            }
        },
        "perf_transform": True,
        "additional_layers": {}
    }
}

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else ("mps" if torch.backends.mps.is_available() else "cpu"))
floating_precision = torch.amp
print(f"Device: {device}")

In [ ]:
# training (with augmentation)
train_transform = transforms.Compose([
    transforms.RandomResizedCrop(224),
    transforms.Grayscale(num_output_channels=3),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(15),
    transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2),
    transforms.GaussianBlur(kernel_size=3),
    transforms.RandomAdjustSharpness(sharpness_factor=2),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

# validation and testing (no augmentation)
val_test_transform = transforms.Compose([
    transforms.Resize(256),
    transforms.Grayscale(num_output_channels=3),
    transforms.CenterCrop(224),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

## Dataset Setup

In [ ]:
dataset_path = os.path.abspath(os.path.join(os.getcwd(), '..', 'datasets', 'combined_ds'))
full_dataset = datasets.ImageFolder(dataset_path, transform=None)

Split the dataset for testing, validation and training. A random seed is used to ensure the split is reproducible.

In [ ]:
train_ratio = 0.7
val_ratio = 0.15
test_ratio = 0.15

dataset_size = len(full_dataset)
train_size = int(train_ratio * dataset_size)
val_size = int(val_ratio * dataset_size)
test_size = dataset_size - train_size - val_size

train_dataset, val_dataset, test_dataset = random_split(
    full_dataset, [train_size, val_size, test_size],
    generator=torch.Generator().manual_seed(42)
)

In [ ]:
class TransformedDataset(Dataset):
    def __init__(self, subset, transform=None):
        self.subset = subset
        self.transform = transform

    def __getitem__(self, index):
        x, y = self.subset[index]
        if self.transform:
            x = self.transform(x)
        return x, y

    def __len__(self):
        return len(self.subset)

In [ ]:
def dataset_loader_init(batch_size, num_workers, perf_transform):
  tr_ds = train_dataset
  val_ds = val_dataset
  test_ds = test_dataset
  if perf_transform:
    tr_ds = TransformedDataset(tr_ds, train_transform)
  else:
    tr_ds = TransformedDataset(tr_ds, val_test_transform)
  val_ds = TransformedDataset(val_ds, val_test_transform)
  test_ds = TransformedDataset(test_ds, val_test_transform)
  train_loader = DataLoader(tr_ds, batch_size=batch_size, shuffle=True, num_workers=num_workers)
  val_loader = DataLoader(val_ds, batch_size=batch_size, shuffle=False, num_workers=num_workers)
  test_loader = DataLoader(test_ds, batch_size=batch_size, shuffle=False, num_workers=num_workers)
  return train_loader, val_loader, test_loader

In [ ]:
def model_init(dropout, freeze_layers, optimizer_alg, learning_rate,
               weight_decay, dropout_prob=None, frozen_layers=None, additional_layers=None):
  model = models.shufflenet_v2_x1_0(weights=ShuffleNet_V2_X1_0_Weights.IMAGENET1K_V1)

  num_features = model.fc.in_features

  if additional_layers is not None:
    for layer in reversed(list(additional_layers)):
      if hasattr(layer, 'out_channels'):
        num_features = layer.out_channels
        break

  fc = model.fc

  if dropout:
      fc_dropout_prob = dropout_prob.get("fc", 0.0)
      fc = nn.Sequential(
          nn.Linear(num_features, 2),
          nn.Dropout(p=fc_dropout_prob)
      )

      new_conv5 = nn.Sequential()
      for layer in model.conv5:
          new_conv5.add_module(layer.__class__.__name__, layer)
          if isinstance(layer, nn.ReLU):
              conv5_dropout_prob = dropout_prob.get("conv5", 0.0)
              new_conv5.add_module("Dropout2d", nn.Dropout2d(p=conv5_dropout_prob))
      model.conv5 = new_conv5

      stages_dropout_prob = dropout_prob.get("stages", {})
      for stage_name, dropout_rate in stages_dropout_prob.items():
          stage_module = getattr(model, stage_name)

          for i, inverted_residual in enumerate(stage_module):
              if hasattr(inverted_residual, 'branch1') and inverted_residual.branch1 is not None:
                  new_branch1 = nn.Sequential()
                  for j, layer in enumerate(inverted_residual.branch1):
                      new_branch1.add_module(f"{j}_{layer.__class__.__name__}", layer)
                      if isinstance(layer, nn.ReLU):
                          new_branch1.add_module(f"{j}_Dropout2d", nn.Dropout2d(p=dropout_rate))

                  inverted_residual.branch1 = new_branch1

              new_branch2 = nn.Sequential()
              for j, layer in enumerate(inverted_residual.branch2):
                  new_branch2.add_module(f"{j}_{layer.__class__.__name__}", layer)
                  if isinstance(layer, nn.ReLU):
                      new_branch2.add_module(f"{j}_Dropout2d", nn.Dropout2d(p=dropout_rate))

              inverted_residual.branch2 = new_branch2
  else:
      fc = nn.Linear(num_features, 2)

  if additional_layers is not None:
      delattr(model, 'fc')
      model.added = additional_layers
      model.pooling = nn.Sequential(
          nn.AdaptiveAvgPool2d(1),
          nn.Flatten()
      )
      model.fc = fc
      def new_forward(self, x):
          x = self.conv1(x)
          x = self.maxpool(x)
          x = self.stage2(x)
          x = self.stage3(x)
          x = self.stage4(x)
          x = self.conv5(x)
          x = self.added(x)
          x = self.pooling(x)
          x = self.fc(x)
          return x
      import types
      model.forward = types.MethodType(new_forward, model)
  else:
      model.fc = fc

  if freeze_layers:
    for layer_name in frozen_layers:
        layer = getattr(model, layer_name)
        for param in layer.parameters():
            param.requires_grad = False

  trainable_params = filter(lambda p: p.requires_grad, model.parameters())
  optimizer = optimizer_alg(trainable_params, lr=learning_rate, weight_decay=weight_decay)

  model = model.to(device)
  return model, optimizer

In [ ]:
def validation_phase(model, device, valid_loader, criterion):
  print("Start Validation")
  model.eval()
  val_running_loss = 0.0
  val_running_corrects = 0

  with torch.no_grad():
    for inputs, labels in valid_loader:
      inputs = inputs.to(device)
      labels = labels.to(device)
      outputs = model(inputs)
      loss = criterion(outputs, labels)
      val_running_loss += loss.item() * inputs.size(0)

      preds = torch.sigmoid(outputs)
      pred_labels = torch.argmax(preds, dim=1)
      val_running_corrects += torch.sum(pred_labels == labels)

  val_epoch_loss = val_running_loss / len(valid_loader.dataset)
  val_epoch_acc = val_running_corrects / len(valid_loader.dataset)

  return val_epoch_loss, val_epoch_acc.item()

In [ ]:
def train_stage(model, train_loader, device, optimizer, criterion):
  print("Start Training Model")
  model.train()
  running_loss = 0.0
  running_corrects = 0

  for inputs, labels in train_loader:
    inputs = inputs.to(device)
    labels = labels.to(device)

    optimizer.zero_grad()
    outputs = model(inputs)
    loss = criterion(outputs, labels)
    loss.backward()
    optimizer.step()

    running_loss += loss.item() * inputs.size(0)
    preds = torch.sigmoid(outputs)
    pred_labels = torch.argmax(preds, dim=1)
    true_labels = labels
    running_corrects += torch.sum(pred_labels == true_labels)

  epoch_loss = running_loss / len(train_loader.dataset)
  epoch_acc = running_corrects / len(train_loader.dataset)

  return epoch_loss, epoch_acc.item()

In [ ]:
def plot_training_curves(train_losses, val_losses, train_accuracies, val_accuracies):
  plt.figure(figsize=(14, 5))
  plt.subplot(1, 2, 1)
  plt.plot(range(1, len(train_losses) + 1), train_losses, 'b-', label='Training Loss')
  plt.plot(range(1, len(val_losses) + 1), val_losses, 'r-', label='Validation Loss')
  plt.xlabel('Epoch')
  plt.ylabel('Loss')
  plt.grid(True, linestyle='--', alpha=0.6)
  plt.legend(loc='upper right')
  plt.title('Training and Validation Loss')

  plt.subplot(1, 2, 2)
  plt.plot(range(1, len(train_accuracies) + 1), train_accuracies, 'b-', label='Training Accuracy')
  plt.plot(range(1, len(val_accuracies) + 1), val_accuracies, 'r-', label='Validation Accuracy')
  plt.xlabel('Epoch')
  plt.ylabel('Accuracy')
  plt.grid(True, linestyle='--', alpha=0.6)
  plt.legend(loc='lower right')
  plt.title('Training and Validation Accuracy')

  plt.tight_layout()
  plt.show()

In [ ]:
def train_model(model, criterion, num_epochs, optimizer, model_name):
  train_losses = []
  train_accuracies = []
  val_losses = []
  val_accuracies = []
  initial_val_loss, initial_val_acc = validation_phase(model, device, val_loader, criterion)
  best_val_loss = initial_val_loss
  best_val_acc = initial_val_acc
  print(f"Initial Validation Loss: {initial_val_loss:.4f}, Validation Accuracy: {initial_val_acc:.4f}")
  for epoch in range(num_epochs):
    epoch_loss, epoch_acc = train_stage(model, train_loader, device, optimizer, criterion)
    train_losses.append(epoch_loss)
    train_accuracies.append(epoch_acc)

    val_epoch_loss,val_epoch_acc = validation_phase(model, device,
                                                    val_loader, criterion)
    val_losses.append(val_epoch_loss)
    val_accuracies.append(val_epoch_acc)

    checkpoints_dir = f"results/{model_name}/checkpoints"
    if not os.path.exists(checkpoints_dir):
      os.makedirs(checkpoints_dir)
    torch.save(model.state_dict(), f"{checkpoints_dir}/{str(epoch)}.pth")

    if val_epoch_loss < best_val_loss and val_epoch_acc > best_val_acc:
      best_val_loss = val_epoch_loss
      best_val_acc = val_epoch_acc
      torch.save(model.state_dict(), f"results/{model_name}/best_model.pth")
      with open(f"results/{model_name}/best_model_epoch.txt", "w") as f:
        f.write(f"The best model epoch - {str(epoch)}")


    print(f"Epoch {epoch+1}/{num_epochs} - "
      f"Train Loss: {epoch_loss:.4f}, Train Acc: {epoch_acc:.4f} | "
      f"Val Loss: {val_epoch_loss:.4f}, Val Acc: {val_epoch_acc:.4f}")
  return train_losses, train_accuracies, val_losses, val_accuracies, model

In [ ]:
for model_name, params in model_params.items():
  num_workers = params["num_workers"]
  batch_size = params["batch_size"]
  perf_transform = params["perf_transform"]
  criterion = getattr(nn, params["criterion"])()
  optimizer_alg = getattr(optim, params["optimizer_alg"])
  learning_rate = params["learning_rate"]
  weight_decay = params["weight_decay"]
  num_epochs = params["num_epochs"]
  dropout = params["dropout"]
  dropout_prob = params["dropout_prob"]
  frozen_layers = params["frozen_layers"]
  freeze_layers = params["freeze_layers"]
  additional_layers_param = params["additional_layers"]
  # Convert empty dictionary to nn.Sequential() if no additional layers are intended
  if isinstance(additional_layers_param, dict) and not additional_layers_param:
      additional_layers = nn.Sequential() # Use an empty sequential module
  else:
      additional_layers = additional_layers_param
      
  print(f"Training model {model_name}")
  train_loader, val_loader, test_loader = dataset_loader_init(batch_size,
                                                              num_workers,
                                                              perf_transform)
  model_name = model_name.replace(" ", "_")
  os.makedirs(f"results/{model_name}", exist_ok=True)
  model, optimizer = model_init(dropout,
                                freeze_layers,
                                optimizer_alg,
                                learning_rate,
                                weight_decay,
                                dropout_prob,
                                frozen_layers,
                                additional_layers)
  print("Start Training")
  train_losses, train_accuracies, val_losses, val_accuracies, model = train_model(model, criterion, num_epochs, optimizer, model_name)
  torch.save(model.state_dict(), f"results/{model_name}/final_model.pth")
  np.save(f"results/{model_name}/train_losses.npy", train_losses)
  np.save(f"results/{model_name}/train_accuracies.npy", train_accuracies)
  np.save(f"results/{model_name}/val_losses.npy", val_losses)
  np.save(f"results/{model_name}/val_accuracies.npy", val_accuracies)

In [ ]:
for model_name, _ in model_params.items():
  train_losses = np.load(f"results/{model_name}/train_losses.npy")
  train_accuracies = np.load(f"results/{model_name}/train_accuracies.npy")
  val_losses = np.load(f"results/{model_name}/val_losses.npy")
  val_accuracies = np.load(f"results/{model_name}/val_accuracies.npy")
  print(f"Model: {model_name}")
  plot_training_curves(train_losses, val_losses, train_accuracies, val_accuracies)

# Compress Model for Export
This block is for compressing the model into a zip file for exporting it and it's training history from Google Colab to a local machine. 

In [ ]:
import shutil

exclude = {}

def delete_checkpoints(exclude):
  for subdir in os.listdir("results"):
    if subdir not in exclude:
      if os.path.exists(f"results/{subdir}/checkpoints"):
        shutil.rmtree(f"results/{subdir}/checkpoints")
    else:
      if os.path.exists(f"results/{subdir}/checkpoints"):
        for file in os.listdir(f"results/{subdir}/checkpoints"):
          if file not in exclude[subdir]:
            os.remove(f"results/{subdir}/checkpoints/{file}")

delete_checkpoints(exclude)

shutil.make_archive('results_conv_size', 'zip', 'results')